# Molar and mass composition of a finite mixture

## Prerequisites

See the [mixture-composition lecture note](../../../docs/lecture-notes/materials/mixtures/molar-and-mass-composition/index.md). This laboratory assumes amount of substance, mass, molar mass, and normalized fractions.

## Objectives

1. Construct an ordered unit-aware molar mixture.
2. Evaluate mole fractions and mixture-average molar mass.
3. Convert the composition to explicit mass units and evaluate mass fractions.
4. Verify a molar-to-mass-to-molar round trip.
5. Visualize why mole fractions and mass fractions generally differ.

The component names and molar masses are synthetic instructional inputs, not material data.


## Definitions

For species amount $n_i$ and molar mass $M_i$,

$$
x_i=\frac{n_i}{\sum_j n_j},
\qquad
m_i=n_iM_i,
\qquad
w_i=\frac{m_i}{\sum_jm_j}.
$$

The equivalent mixture-average molar-mass expressions are

$$
\overline M=\sum_i x_iM_i
=\left(\sum_i\frac{w_i}{M_i}\right)^{-1}.
$$


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.materials.mixtures.molar import (
    MassToMolarMixtureConversionRequest,
    MixtureCompositionConverter,
    MolarMixture,
    MolarToMassMixtureConversionRequest,
)
from projectkoios.physkit.units import PhysicalUnit, VectorQuantity

In [ ]:
species = ("component A", "component B")
source_mixture = MolarMixture(
    species=species,
    mole_amounts=VectorQuantity(
        magnitude=np.asarray([1.0, 3.0], dtype=np.float64),
        unit=PhysicalUnit(expression="mole"),
    ),
    molar_masses=VectorQuantity(
        magnitude=np.asarray([20.0, 50.0], dtype=np.float64),
        unit=PhysicalUnit(expression="gram / mole"),
    ),
)
converter = MixtureCompositionConverter()
mass_conversion = converter.convert_to_mass(
    request=MolarToMassMixtureConversionRequest(
        mixture=source_mixture,
        target_mass_unit=PhysicalUnit(expression="gram"),
    )
)
mass_mixture = mass_conversion.mixture
round_trip = converter.convert_to_molar(
    request=MassToMolarMixtureConversionRequest(
        mixture=mass_mixture,
        target_amount_unit=PhysicalUnit(expression="millimole"),
    )
)

In [ ]:
assert np.allclose(
    source_mixture.mole_fractions.magnitude,
    np.asarray([0.25, 0.75], dtype=np.float64),
)
assert np.allclose(
    mass_mixture.mass_amounts.magnitude,
    np.asarray([20.0, 150.0], dtype=np.float64),
)
assert np.isclose(np.sum(mass_mixture.mass_fractions.magnitude), 1.0)
assert np.isclose(
    source_mixture.mean_molar_mass.magnitude,
    mass_mixture.mean_molar_mass.magnitude,
)
assert np.allclose(
    round_trip.mixture.mole_amounts.magnitude,
    np.asarray([1000.0, 3000.0], dtype=np.float64),
)

In [ ]:
component_indices = np.arange(len(species), dtype=np.float64)
bar_width = 0.36
figure, axis = plt.subplots(figsize=(8.0, 4.5))
axis.bar(
    component_indices - bar_width / 2.0,
    source_mixture.mole_fractions.magnitude,
    width=bar_width,
    label="mole fraction",
)
axis.bar(
    component_indices + bar_width / 2.0,
    mass_mixture.mass_fractions.magnitude,
    width=bar_width,
    label="mass fraction",
)
axis.set(
    xticks=component_indices,
    xticklabels=species,
    ylabel="fraction",
    title="Equivalent molar and mass representations",
)
axis.set_ylim(0.0, 1.0)
axis.grid(True, axis="y")
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The heavier synthetic component has a larger mass fraction than mole fraction. Both representations describe the same finite composition: conversion changes the represented extensive quantities and normalized fractions, not species ordering or identity. The two mean-molar-mass formulas agree after conversion.

## Exercises

1. Exchange the two molar masses while preserving mole amounts and explain the fraction changes.
2. Add a zero-amount third component and verify that the positive total remains valid.
3. Convert the output masses to kilograms instead of grams.
4. Determine when the mole-fraction and mass-fraction vectors are identical.
